# CMSC 173 &middot; Machine Learning &mdash; Week 2 Lab
## Parameter Estimation: Method of Moments vs Maximum Likelihood

Last week's lecture asked a plain question: given some data, how do you recover the
settings ("parameters") of the distribution that produced it? You saw two answers &mdash;
**Method of Moments (MoM)** and **Maximum Likelihood (MLE)**. This lab makes both concrete
by *running* them on data you generate yourself, so you can watch where they agree, where
they differ, and why MLE is the one the rest of the course leans on.

You will mostly **run a cell and notice what happens**, then jot a sentence. There is very
little to derive by hand &mdash; the code does the maths; your job is to read the output.

**Not graded.** About 45 minutes. NumPy only &mdash; no scikit-learn yet (that arrives in week 4,
after you have fit linear regression by hand in week 3).

---
## Part 0 &middot; Setup

Run this. Same seed as week 1, so your "random" numbers match mine.

In [ ]:
import sys
import numpy as np

print("Python", sys.version.split()[0])
print("NumPy ", np.__version__)
rng = np.random.default_rng(173)
print("\nReady.")

---
## Part 1 &middot; A moment is just an average of a power

That's the whole idea, so don't let the word scare you. Add up the data and divide by $n$:
that's the **mean**. Add up the *squared* data and divide by $n$: that's the second "moment".
Method of Moments is built on one move &mdash; the averages you compute *from your data* should
match the averages the distribution *predicts*. No new machinery, just averages.

Run the cell and look at the last two numbers.

In [ ]:
data = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])

mean = data.mean()                    # average of the data
var  = ((data - mean)**2).mean()      # average squared distance from the mean = variance
m2   = (data**2).mean()               # average of the SQUARED data (second moment)

print("mean          =", mean)
print("variance      =", round(var, 4))
print("m2 - mean**2  =", round(m2 - mean**2, 4))   # look: same as the variance

**Answer here** (double-click to edit):

1. `variance` and `m2 - mean**2` printed the same number &mdash; that's a handy shortcut, not a
   coincidence. You don't need to prove it; just say when computing the variance as
   `m2 - mean**2` might be more convenient than the direct way.
   &rarr; *your answer*

2. `var` above divides by $n$. NumPy's `data.var(ddof=1)` divides by $n-1$. Print both. Which
   is bigger, and does the gap look big or tiny for these 8 points?
   &rarr; *your answer*

---
## Part 2 &middot; Method of Moments, on a Normal

Now run it the other way: start from data whose *true* settings you know, and see how close
the estimates land. For a Normal, MoM is refreshingly blunt &mdash; the estimate of the mean is
just the sample mean, and the estimate of the variance is just the sample variance. Because
we generate the data, we can check the estimates against the truth.

In [ ]:
true_mu, true_sigma = 5.0, 2.0
sample = rng.normal(true_mu, true_sigma, size=200)   # 200 points from a known Normal

def mom_normal(x):
    """Method-of-Moments estimate for a Normal: (mean, variance). Just two averages."""
    mu_hat  = x.mean()
    var_hat = ((x - mu_hat)**2).mean()
    return mu_hat, var_hat

mu_hat, var_hat = mom_normal(sample)
print(f"true:     mu = {true_mu}, variance = {true_sigma**2:.2f}")
print(f"estimate: mu = {mu_hat:.3f}, variance = {var_hat:.3f}")

**Answer here:**

1. Change `size=200` to `size=20`, run, then change it back. Do the estimates land closer to
   the truth or further away? State the rule in one sentence.
   &rarr; *your answer*

2. Here MoM behaved perfectly. But recall the **Gamma** example from lecture &mdash; name one way a
   Method-of-Moments estimate can come out as a value that doesn't even make sense.
   &rarr; *your answer*

---
## Part 3 &middot; Maximum Likelihood (the course's default)

MoM matched averages. Maximum Likelihood asks something more direct:

> Of all the settings I *could* choose, which one makes the data I actually observed the
> **most probable**?

The **likelihood** is just "how probable is my whole dataset under this guess?" &mdash; bigger
means a better guess. One practical tweak: instead of *multiplying* every point's probability
(hundreds of tiny numbers multiply down to zero on a computer), we **add up their logs**. So
we maximise the **log-likelihood** = the sum of log-probabilities.

The punchline for the Normal: the settings that maximise the likelihood are the **same** mean
and variance MoM already gave you. Let's just *watch* that, not derive it.

In [ ]:
def normal_loglik(x, mu, var):
    """How probable is x under a Normal(mu, var)? Return the summed log-probability."""
    n = len(x)
    # this is just log of the bell-curve formula, added up over all the points
    return -0.5 * n * np.log(2 * np.pi * var) - ((x - mu)**2).sum() / (2 * var)

# Try 401 candidate means; keep whichever scores the highest log-likelihood.
candidates = np.linspace(4.0, 6.0, 401)
scores = np.array([normal_loglik(sample, m, var_hat) for m in candidates])
best_mu = candidates[scores.argmax()]

print(f"best mu by just trying lots of values = {best_mu:.3f}")
print(f"the sample mean (what MoM gave)       = {sample.mean():.3f}")
print("same answer, two different roads.")

**Answer here:**

1. We *added logs* instead of *multiplying probabilities*. In your own words, what goes wrong
   if you multiply a few hundred tiny probabilities directly on a computer?
   &rarr; *your answer*

2. Here "try lots of values" and the exact answer agreed. Later some models have **no** exact
   formula and you must search like this. Name one (a guess is fine).
   &rarr; *your answer*

---
## Part 4 &middot; Is the estimate any good?

Run the same recipe on different samples and you get slightly different answers &mdash; an
estimate is a moving target. Two fair questions, in plain words:

- is it **wrong on average**? (statisticians call that *bias*)
- does it **home in on the truth as you collect more data**? (*consistency*)

No theory needed &mdash; just try it thousands of times and watch the averages.

In [ ]:
true_var = true_sigma**2
sizes  = [10, 50, 500]
trials = 2000

print(f"the true variance is {true_var:.2f}\n")
print(f"{'n':>5} {'avg estimate (/n)':>18} {'avg estimate (/(n-1))':>22}")
for n in sizes:
    a, b = [], []
    for _ in range(trials):
        s = rng.normal(true_mu, true_sigma, size=n)
        a.append(((s - s.mean())**2).mean())   # divide by n
        b.append(s.var(ddof=1))                # divide by n-1
    print(f"{n:>5} {np.mean(a):>18.3f} {np.mean(b):>22.3f}")

**Answer here:**

1. The `/n` column sits a little *below* the true 4.0; the `/(n-1)` column sits right on it.
   That small undershoot is the *bias*. Look at the `n = 10` row &mdash; is the undershoot big or
   tiny?
   &rarr; *your answer*

2. Both columns get closer to 4.0 as $n$ grows. In one sentence: why does that make people
   comfortable using the slightly-biased `/n` version most of the time?
   &rarr; *your answer*

---
## Part 5 &middot; When both methods give the exact same answer

For a **Poisson** &mdash; counts of things, like dengue cases per barangay per week, or typos per
page &mdash; MoM and MLE collapse to the *same* simple rule: the estimate of the rate is just the
average count. Confirm it, and notice how all the machinery reduces to "take the mean" once
the distribution is simple enough.

In [ ]:
true_lambda = 3.5
counts = rng.poisson(true_lambda, size=300)

estimate = counts.mean()   # both MoM and MLE give exactly this for a Poisson

print(f"true rate     = {true_lambda}")
print(f"estimate      = {estimate:.3f}   (this is what BOTH methods return)")

**Answer here:**

1. For the Poisson, MoM and MLE are identical. From lecture, name one distribution where they
   are **not** &mdash; and which of the two you'd trust more there.
   &rarr; *your answer*

2. A Poisson only fits non-negative whole-number counts. Give one real Philippine dataset that
   is genuinely count-like, and one that *looks* count-like but would mislead a Poisson model.
   &rarr; *your answer*

---
## Part 6 &middot; Where you actually are

Same as last week &mdash; set the pace honestly.

Replace each `-` with one of: **solid** / **rusty** / **never really got it**.

| | You |
|---|---|
| Mean and variance in NumPy | - |
| The idea of a "moment" | - |
| What a likelihood means (in words) | - |
| Bias vs consistency of an estimate | - |
| Reading a small simulation table | - |

**Which part took longest, and where did you get stuck?**
&rarr; *your answer*

**In one sentence and in plain words: why does this course prefer MLE over MoM by default?**
&rarr; *your answer*

---
## Stretch &mdash; optional

The required part is done; nothing below is graded. Try one if you're curious.

### Stretch 1 &middot; How sure are you? (bootstrap)

You reported one estimate. How much would it wobble on a *different* sample? The **bootstrap**
is a trick: resample your own data (with replacement) many times, re-estimate each time, and
look at the spread. This one is done for you &mdash; read it and run it.

In [ ]:
B = 2000
wobble = np.empty(B)
for i in range(B):
    resample  = rng.choice(counts, size=len(counts), replace=True)
    wobble[i] = resample.mean()          # re-estimate on each fake sample

print(f"estimate              = {counts.mean():.3f}")
print(f"typical wobble (std)  = {wobble.std():.3f}")
print(f"~95% range            = [{np.percentile(wobble, 2.5):.3f}, {np.percentile(wobble, 97.5):.3f}]")

### Stretch 2 &middot; Your turn: the Exponential

Waiting times are often **Exponential**: think of minutes between jeepneys at a stop. The MLE
rule is short &mdash; the estimated rate is `1 / (average wait)`. Data is below. Write the one line
that estimates the rate, then print the truth next to your estimate.

In [ ]:
true_rate = 0.5
waits = rng.exponential(1 / true_rate, size=250)   # numpy uses the mean = 1/rate

# your code here: estimate the rate as 1 / (average wait), then print truth vs estimate


---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to download.

You need a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, then paste it when the cell asks. The cell hides what you type, so
the token never gets saved inside your notebook.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 2

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/2/submit"
    )

# The LIVE notebook, including edits you have not saved yet.
nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]

token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 2 submission page](https://portal.latarak.com/course/cmsc173/lab/2/submit) and upload it.

Blank cells are fine and guesses are fine. What is not useful is polishing this until it hides
what you actually knew &mdash; that just moves the surprise to a later week, where it costs more.